# Retrieval e Embeddings

Nesta aula vamos construir a base de um fluxo de **RAG (Retrieval-Augmented Generation)** usando Python, LangChain, Ollama, Qwen 3 4B, embeddings e Chroma.

Fluxo:

```text
TXT → Chunks → Embeddings → Vector Store → Retriever → Contexto → Qwen → Resposta
```

O objetivo é entender como uma aplicação pode buscar informações relevantes em uma base própria antes de pedir ao LLM para responder.


## 1. Retrieval

Retrieval é o processo de buscar informações relevantes em uma base de documentos.

Em vez de colocar toda a documentação no prompt:

```text
Pergunta
   ↓
Retriever
   ↓
Trechos relevantes
   ↓
LLM
   ↓
Resposta
```

Isso é uma das bases de aplicações RAG.


## 2. O que são embeddings?

Embeddings transformam textos em vetores numéricos.

```text
"organizar tarefas"
       ↓
[0.12, -0.44, 0.81, ...]
```

Textos semanticamente semelhantes tendem a possuir vetores próximos. Isso permite realizar buscas por similaridade.


## 3. Embeddings x LLM

São funções diferentes:

**Modelo de embeddings**
```text
Texto → vetor
```

**LLM**
```text
Pergunta + contexto → resposta
```

Neste projeto:
- `nomic-embed-text` → embeddings;
- `qwen3:4b` → geração da resposta.


## 4. Preparando o modelo de embeddings

Baixe o modelo no Ollama:

```powershell
docker exec -it langchain-ollama ollama pull nomic-embed-text
```

Para esta aula, adicione ao `requirements.txt`:

```text
langchain-chroma
langchain-text-splitters
chromadb
```

Depois execute:

```powershell
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma


## 5. Carregando a base de conhecimento

O arquivo `nexa_ai_base.txt` representa nossa pequena base de conhecimento.

Em um projeto real, a fonte poderia ser PDF, Markdown, banco de dados, documentação interna ou páginas web.


In [ ]:
base_path = Path("nexa_ai_base.txt")
texto = base_path.read_text(encoding="utf-8")

documento = Document(
    page_content=texto,
    metadata={"source": base_path.name}
)

print(documento.metadata)


## 6. Dividindo em chunks

Documentos grandes são normalmente divididos em pequenos trechos.

```text
Documento
   ↓
Chunk 1
Chunk 2
Chunk 3
...
```

Isso permite recuperar somente as partes relevantes.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

documentos = splitter.split_documents([documento])

print("Quantidade de chunks:", len(documentos))
print(documentos[0].page_content)


## 7. `chunk_size` e `chunk_overlap`

`chunk_size` controla aproximadamente o tamanho de cada trecho.

`chunk_overlap` mantém uma parte do trecho anterior no próximo chunk.

O overlap ajuda a evitar que informações importantes sejam cortadas exatamente na fronteira entre dois chunks.


## 8. Criando os embeddings

Vamos utilizar `nomic-embed-text` através do Ollama, portanto os embeddings também são gerados localmente.


In [ ]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)


## 9. Gerando um embedding

`embed_query()` transforma uma consulta em vetor.


In [ ]:
texto_teste = "Como a Nexa AI ajuda a organizar tarefas?"

vetor = embeddings.embed_query(texto_teste)

print("Tamanho do vetor:", len(vetor))
print("Primeiros valores:", vetor[:10])


## 10. Vector Store

Precisamos armazenar os chunks junto com seus embeddings.

Neste estudo utilizaremos **Chroma**:

```text
Chunk + Embedding + Metadata
             ↓
           Chroma
```


In [ ]:
vector_store = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    collection_name="nexa-ai-estudo"
)


## 11. Criando o Retriever

O retriever realiza a busca pelos documentos mais semelhantes à consulta.

Com `k=3`, recuperamos os três chunks mais relevantes.


In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


## 12. Testando somente o Retrieval

Antes de envolver o LLM, vamos observar quais trechos são encontrados.


In [ ]:
pergunta = "O que a Central IA pode fazer?"

documentos_relevantes = retriever.invoke(pergunta)

for i, doc in enumerate(documentos_relevantes, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)


## 13. Retrieval + LLM

Agora conectamos as duas partes:

```text
Pergunta
   ↓
Retriever
   ↓
Contexto
   ↓
Prompt
   ↓
Qwen 3 4B
   ↓
Resposta
```


In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

prompt = PromptTemplate.from_template(
    '''
    Responda à pergunta utilizando somente o contexto fornecido.

    Se a resposta não estiver no contexto, diga que a informação
    não está disponível na base de conhecimento.

    Contexto:
    {context}

    Pergunta:
    {question}
    '''
)


## 14. Formatando os documentos

O retriever retorna objetos `Document`. Vamos juntar o conteúdo deles em uma string para colocar no prompt.


In [ ]:
def format_documents(docs):
    return "\n\n".join(doc.page_content for doc in docs)


## 15. Montando a chain com LCEL

O dicionário abaixo cria dois caminhos:

```text
pergunta → retriever → format_documents → context
pergunta → question
```

Depois ambos entram no prompt e o resultado segue para o Qwen.


In [ ]:
chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


## 16. Fazendo perguntas

Agora temos uma pequena implementação de RAG.


In [ ]:
for pergunta in [
    "Quais funcionalidades existem na Central IA?",
    "Qual banco de dados a Nexa AI utiliza?"
]:
    resposta = chain.invoke(pergunta)
    print("\nPergunta:", pergunta)
    print("Resposta:", resposta)


## 17. Fluxo completo

```text
nexa_ai_base.txt
       ↓
Document
       ↓
Chunking
       ↓
nomic-embed-text
       ↓
Chroma
       ↓
Retriever
       ↓
Chunks relevantes
       ↓
Prompt + Pergunta
       ↓
Qwen 3 4B
       ↓
Resposta
```

Esse é o fundamento de um pipeline RAG.


## 18. Retrieval, Embeddings e RAG

- **Embeddings:** representam textos como vetores.
- **Retrieval:** encontra informações relevantes.
- **Vector Store:** armazena e permite consultar os vetores.
- **RAG:** combina recuperação de contexto com geração do LLM.

De forma simplificada:

```text
RAG = Retrieval + Generation
```
